# Instancia 1 del Poster
## Estadisticas de Nigeria

Constanza Efkhanian, Julian Notario y Maria Florencia Pascale

#### Fuente a analizar: Nigeria General Household Survey
https://microdata.worldbank.org/catalog/6410/study-description

Organizacion encargada de recolectar datos: National Bureau of Statistics, afiliada con el gobierno federal de Nigeria

### Bibliotecas

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

### Paths

In [2]:
BASE = "/Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/cvs"
PP = BASE
PH = BASE
OUT = "/Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/diagnostico"

os.makedirs(OUT, exist_ok=True)

### Estilo global
Principios de Schwabish (2014): reducir clutter, mostrar los datos, integrar texto.

In [3]:
# Paleta de colores
PALETTE   = "#2C3E50"
ACCENT    = "#E74C3C"
SECONDARY = "#3498DB"
GRAY      = "#95A5A6"

# Schwabish (2014): grillas suaves, sin marcos innecesarios
sns.set_theme(style="ticks", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor"  : "white",
    "axes.facecolor"    : "white",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.spines.left"  : True,
    "axes.spines.bottom": True,
    "axes.grid"         : True,
    "grid.color"        : "#E8E8E8",
    "grid.linewidth"    : 0.6,
    "axes.linewidth"    : 0.8,
    "xtick.major.size"  : 3,
    "ytick.major.size"  : 3,
    "font.family"       : "sans-serif",
    "axes.labelsize"    : 10,
    "axes.titlesize"    : 11,
    "figure.titlesize"  : 13,
})

def save(fig, name):
    path = f"{OUT}/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  v {name}.png")

def yn_to_bin(s):
    """Convierte '1. YES' / '2. NO' a 1 / 0."""
    return s.map({"1. YES": 1, "2. NO": 0})

def label_bars_h(ax, fmt="{:.1f}%", pad=0.4, fontsize=9):
    """Etiquetas directas en barras horizontales (Schwabish: integrar texto)."""
    for bar in ax.patches:
        w = bar.get_width()
        if abs(w) > 0:
            ax.text(w + pad, bar.get_y() + bar.get_height() / 2,
                    fmt.format(w), va="center", ha="left", fontsize=fontsize)

def label_bars_v(ax, fmt="{:.1f}%", pad=1, fontsize=9):
    """Etiquetas directas en barras verticales."""
    for bar in ax.patches:
        h = bar.get_height()
        if abs(h) > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + pad,
                    fmt.format(h), va="bottom", ha="center",
                    fontsize=fontsize, fontweight="bold")

### PASO 0 - Cargar archivos con columnas seleccionadas

In [4]:
print("[0] Cargando archivos...")

# -- Demografia individual (sect1) -------------------------
DEMO_COLS = ["hhid", "indiv", "zone",
             "s1q2",   # sexo (MALE/FEMALE)
             "s1q6",   # edad
             "s1q8",   # casado (YES/NO)
             "s1q25",  # actualmente en la escuela
             "s1q28",  # nivel educativo mas alto
             "s1q29",  # sector de empleo
             ]
demo = pd.read_csv(f"{PP}/sect1_plantingw5.csv",
                   usecols=DEMO_COLS, low_memory=False)

# -- Target planting (sect1d) ------------------------------
TARGET_COLS_PP = ["hhid", "indiv", "FILTER",
                  "s1dq1",  # desea migrar
                  "s1dq3a", # destino: dentro del pais
                  "s1dq3b", # destino: fuera del pais
                  "s1dq5",  # probabilidad de migrar
                  "s1dq7",  # intento migrar antes
                  ]
target_pp = pd.read_csv(f"{PP}/sect1d_plantingw5.csv",
                        usecols=TARGET_COLS_PP, low_memory=False)

# -- Remesas (sect1e) --------------------------------------
REMIT_COLS = ["hhid", "indiv",
              "s1eq0a", # recibe remesas del exterior
              "s1eq0b", # recibe remesas de Nigeria
              "s1eq3",  # monto del exterior
              "s1eq6",  # monto interno
              ]
remit = pd.read_csv(f"{PP}/sect1e_plantingw5.csv",
                    usecols=REMIT_COLS, low_memory=False)

# -- Mercado laboral (sect4a) ------------------------------
LABOR_COLS = ["hhid", "indiv",
              "s4aq1",  # trabajo en ultimos 7 dias
              "s4aq4",  # busco trabajo
              "s4aq46", # horas trabajadas por semana
              "s4aq47", # ingresos laborales
              ]
labor = pd.read_csv(f"{PP}/sect4a_plantingw5.csv",
                    usecols=LABOR_COLS, low_memory=False)

# -- Seguridad alimentaria (sect9, nivel hogar) -----------
FOOD_COLS = ["hhid",
             "s9q1a", "s9q1b", "s9q1c", "s9q1d", "s9q1e",
             "s9q1f", "s9q1g", "s9q1h", "s9q1i", "s9q1j",
             "s9q2a", # HFIAS score
             "s9q4",  # dias sin comer
             ]
food = pd.read_csv(f"{PP}/sect9_plantingw5.csv",
                   usecols=FOOD_COLS, low_memory=False)

# -- Pesos muestrales (secta) ------------------------------
WEIGHTS_COLS = ["hhid", "wt_cross_wave5", "sector"]
weights = pd.read_csv(f"{PP}/secta_plantingw5.csv",
                      usecols=WEIGHTS_COLS, low_memory=False)
weights = weights.rename(columns={"sector": "sector_hh"})

# -- Target harvest (sect3b) -------------------------------
TARGET_COLS_PH = ["hhid", "indiv", "s3bq1", "PPMIGASP_prefilled"]
target_ph = pd.read_csv(f"{PH}/sect3b_harvestw5.csv",
                        usecols=TARGET_COLS_PH, low_memory=False)

# -- Shocks economicos (sect12) ----------------------------
shocks_raw = pd.read_csv(f"{PH}/sect12_harvestw5.csv",
                         low_memory=False)

print("  Archivos cargados correctamente.")

[0] Cargando archivos...
  Archivos cargados correctamente.


### SECCION 1 - Estructura de archivos

In [5]:
print("[1] Estructura de archivos...")

file_info = {
    "sect1d (target planting)": target_pp,
    "sect1 (demografia)"      : demo,
    "sect1e (remesas)"         : remit,
    "sect4a (mercado laboral)" : labor,
    "sect9 (seg. alimentaria)" : food,
    "secta (pesos)"            : weights,
    "sect3b (target harvest)"  : target_ph,
    "sect12 (shocks)"          : shocks_raw,
}

names = list(file_info.keys())
rows  = [df.shape[0] for df in file_info.values()]
cols_ = [df.shape[1] for df in file_info.values()]

# Schwabish: barras horizontales, etiquetas directas, grilla suave
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Seccion 1 - Estructura de los archivos de entrada", fontweight="bold")

for ax, values, xlabel, title, color in [
    (axes[0], rows,  "Filas",     "Observaciones por archivo",  SECONDARY),
    (axes[1], cols_, "Variables", "Variables seleccionadas",    PALETTE),
]:
    bars = ax.barh(names, values, color=color, alpha=0.85, height=0.55)
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontweight="bold")
    ax.invert_yaxis()
    ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)
    for bar, v in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{v:,}", va="center", fontsize=9)

plt.tight_layout()
save(fig, "01_estructura_archivos")

[1] Estructura de archivos...
  v 01_estructura_archivos.png


### SECCION 2 - Factibilidad del merge

In [6]:
print("[2] Factibilidad del merge...")

indiv_files = {"sect1d": target_pp, "sect1": demo,
               "sect1e": remit, "sect4a": labor, "sect3b": target_ph}
hh_files    = {"sect9": food, "secta": weights}

target_indivs = set(zip(
    target_pp[target_pp["FILTER"] == "1. YES"]["hhid"],
    target_pp[target_pp["FILTER"] == "1. YES"]["indiv"]
))
hh_target = set(target_pp[target_pp["FILTER"] == "1. YES"]["hhid"])

merge_stats = []
for name, df in indiv_files.items():
    key     = set(zip(df["hhid"], df["indiv"]))
    overlap = len(key & target_indivs)
    merge_stats.append({
        "Archivo"         : name,
        "Nivel"           : "Individuo",
        "Unidades unicas" : len(key),
        "Overlap c/target": overlap,
        "% cobertura"     : round(overlap / len(target_indivs) * 100, 1),
    })
for name, df in hh_files.items():
    hh_file  = set(df["hhid"])
    overlap  = len(hh_file & hh_target)
    merge_stats.append({
        "Archivo"         : name,
        "Nivel"           : "Hogar",
        "Unidades unicas" : len(hh_file),
        "Overlap c/target": overlap,
        "% cobertura"     : round(overlap / len(hh_target) * 100, 1),
    })

merge_df = pd.DataFrame(merge_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 4),
                         gridspec_kw={"width_ratios": [2, 1]})
fig.suptitle(
    "Seccion 2 - Factibilidad del merge\n"
    "(universo: individuos con FILTER=YES en sect1d)",
    fontweight="bold")

ax_tbl = axes[0]
ax_tbl.axis("off")
tbl = ax_tbl.table(cellText=merge_df.values, colLabels=merge_df.columns,
                   loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)
for j in range(len(merge_df.columns)):
    tbl[0, j].set_facecolor(PALETTE)
    tbl[0, j].set_text_props(color="white", fontweight="bold")
for i in range(1, len(merge_df) + 1):
    pct = float(merge_df["% cobertura"].iloc[i - 1])
    color = "#27AE60" if pct > 95 else ("#F39C12" if pct > 80 else "#E74C3C")
    tbl[i, 4].set_facecolor(color)
    tbl[i, 4].set_text_props(color="white", fontweight="bold")

ax_bar = axes[1]
colors_cov = ["#27AE60" if p > 95 else ("#F39C12" if p > 80 else "#E74C3C")
              for p in merge_df["% cobertura"]]
bars = ax_bar.barh(merge_df["Archivo"], merge_df["% cobertura"],
                   color=colors_cov, alpha=0.85, height=0.55)
ax_bar.set_xlim(0, 110)
ax_bar.axvline(100, ls="--", color=GRAY, linewidth=0.8)
ax_bar.set_xlabel("% cobertura")
ax_bar.set_title("Cobertura del merge", fontweight="bold")
ax_bar.invert_yaxis()
ax_bar.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax_bar.grid(axis="y", visible=False)
ax_bar.spines["left"].set_visible(False)
ax_bar.tick_params(left=False)
for bar, p in zip(bars, merge_df["% cobertura"]):
    ax_bar.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                f"{p}%", va="center", fontsize=9)

plt.tight_layout()
save(fig, "02_merge_feasibility")

[2] Factibilidad del merge...
  v 02_merge_feasibility.png


### SECCION 3 - Variable target

In [7]:
print("[3] Variable target...")

tgt = target_pp[target_pp["FILTER"] == "1. YES"].copy()
tgt["target"] = yn_to_bin(tgt["s1dq1"])

tgt_demo = tgt.merge(
    demo[["hhid", "indiv", "zone", "s1q2", "s1q6", "s1q28"]],
    on=["hhid", "indiv"], how="left"
)

# Sector: usar s1q29 si existe
if "s1q29" in demo.columns:
    tgt_demo = tgt_demo.merge(
        demo[["hhid", "indiv", "s1q29"]].rename(columns={"s1q29": "sector"}),
        on=["hhid", "indiv"], how="left"
    )
else:
    tgt_demo["sector"] = "N/D"

tgt_demo["age_group"] = pd.cut(
    tgt_demo["s1q6"],
    bins=[14, 24, 34, 44, 54, 120],
    labels=["15-24", "25-34", "35-44", "45-54", "55+"]
)

global_rate = tgt_demo["target"].mean() * 100

# Schwabish (2014): 5 paneles SIN grafico de torta.
# Los graficos de torta dificultan la comparacion de areas y angulos;
# se reemplaza por barras horizontales con etiquetas directas.

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    f"Seccion 3 - Aspiraciones migratorias: "
    f"{global_rate:.1f}% de los adultos 15+ desea migrar",
    fontweight="bold")
axes = axes.flatten()

COLORS_T = {"Desea migrar": SECONDARY, "No desea": ACCENT}

# 3.1 Distribucion global - barras horizontales (reemplaza pie chart)
ax = axes[0]
pcts     = [tgt["target"].mean() * 100, (1 - tgt["target"].mean()) * 100]
labels_g = ["Desea migrar", "No desea migrar"]
bars = ax.barh(labels_g, pcts, color=[SECONDARY, ACCENT], alpha=0.88, height=0.45)
ax.set_xlim(0, 110)
ax.set_xlabel("% de adultos 15+")
ax.set_title(f"Distribucion global (n={len(tgt):,})", fontweight="bold")
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for bar, p in zip(bars, pcts):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{p:.1f}%", va="center", fontsize=12, fontweight="bold")

# 3.2 Por sexo
ax = axes[1]
sex_rate = (tgt_demo.groupby("s1q2")["target"]
            .mean().reset_index()
            .rename(columns={"s1q2": "Sexo", "target": "Tasa_YES"}))
bars = ax.bar(sex_rate["Sexo"], sex_rate["Tasa_YES"] * 100,
              color=[SECONDARY, ACCENT], alpha=0.88, width=0.45)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1,
           label=f"Media global ({global_rate:.1f}%)")
ax.set_ylim(0, 100)
ax.set_ylabel("% desea migrar")
ax.set_title("Tasa de aspiracion por sexo", fontweight="bold")
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
label_bars_v(ax)

# 3.3 Por zona geografica
ax = axes[2]
zone_rate   = tgt_demo.groupby("zone")["target"].mean().sort_values()
short_zones = [str(z).split(". ")[-1] for z in zone_rate.index]
bars = ax.barh(short_zones, zone_rate.values * 100,
               color=PALETTE, alpha=0.82, height=0.55)
ax.axvline(global_rate, ls="--", color=ACCENT, linewidth=1, alpha=0.7)
ax.set_xlabel("% desea migrar")
ax.set_title("Tasa por zona geografica", fontweight="bold")
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
label_bars_h(ax)

# 3.4 Por grupo de edad - panel unico, sin eje doble
# (Schwabish: evitar ejes dobles; incluir n dentro de la barra)
ax = axes[3]
age_rate = (tgt_demo.groupby("age_group", observed=True)["target"]
            .agg(["mean", "count"]))
bars = ax.bar(age_rate.index.astype(str), age_rate["mean"] * 100,
              color=SECONDARY, alpha=0.82, width=0.55)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1)
ax.set_ylabel("% desea migrar")
ax.set_title("Tasa de aspiracion por grupo de edad", fontweight="bold")
ax.set_ylim(0, 100)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
label_bars_v(ax)
for bar, (_, row) in zip(bars, age_rate.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, 2,
            f"n={row['count']:,}", ha="center", va="bottom",
            fontsize=7.5, color="white", fontweight="bold")

# 3.5 Por sector (urbano/rural)
ax = axes[4]
sec_rate = (tgt_demo.groupby("sector")["target"]
            .mean().reset_index()
            .rename(columns={"sector": "Sector", "target": "Tasa"}))
bars = ax.bar(sec_rate["Sector"].astype(str), sec_rate["Tasa"] * 100,
              color=[PALETTE, SECONDARY], alpha=0.88, width=0.4)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1)
ax.set_ylim(0, 100)
ax.set_ylabel("% desea migrar")
ax.set_title("Tasa por sector (urbano / rural)", fontweight="bold")
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
label_bars_v(ax)

# 3.6 Transicion planting -> harvest
ax = axes[5]
both = target_pp[target_pp["FILTER"] == "1. YES"].merge(
    target_ph[["hhid", "indiv", "s3bq1"]], on=["hhid", "indiv"], how="inner"
)
both["pp"] = yn_to_bin(both["s1dq1"])
both["ph"] = yn_to_bin(both["s3bq1"])
trans = pd.crosstab(
    both["pp"].map({1: "Si (planting)", 0: "No (planting)"}),
    both["ph"].map({1: "Si (harvest)",  0: "No (harvest)"}),
    normalize="index"
) * 100
sns.heatmap(trans, annot=True, fmt=".1f", cmap="Blues",
            ax=ax, cbar=False, linewidths=0.4, linecolor="white",
            annot_kws={"size": 11, "weight": "bold"})
ax.set_title("Transicion aspiraciones\nplanting > harvest (%)", fontweight="bold")
ax.set_ylabel("Planting")
ax.set_xlabel("Harvest")
ax.text(0, -0.18, "Valores: % de fila",
        transform=ax.transAxes, fontsize=8, color=GRAY)

plt.tight_layout()
save(fig, "03_target_variable")

[3] Variable target...
  v 03_target_variable.png


### SECCION 4 - Dataset fusionado: missings y correlaciones

In [8]:
print("[4] Dataset fusionado y missings...")

base = tgt[["hhid", "indiv", "target",
            "s1dq3a", "s1dq3b", "s1dq5", "s1dq7"]].copy()
base = base.merge(
    demo[["hhid", "indiv", "zone", "s1q2", "s1q6", "s1q8", "s1q25", "s1q28"]],
    on=["hhid", "indiv"], how="left")
base = base.merge(
    remit[["hhid", "indiv", "s1eq0a", "s1eq0b", "s1eq3", "s1eq6"]],
    on=["hhid", "indiv"], how="left")
base = base.merge(
    labor[["hhid", "indiv", "s4aq1", "s4aq4", "s4aq46", "s4aq47"]],
    on=["hhid", "indiv"], how="left")
base = base.merge(food, on="hhid", how="left")
base = base.merge(weights[["hhid", "wt_cross_wave5"]], on="hhid", how="left")
base = base.merge(
    target_ph[["hhid", "indiv", "s3bq1"]].rename(columns={"s3bq1": "target_harvest"}),
    on=["hhid", "indiv"], how="left")

print(f"  Dataset fusionado: {base.shape[0]:,} filas x {base.shape[1]} columnas")

def tag(col):
    if col in ["hhid", "indiv", "target", "target_harvest"]: return "IDs / Target"
    if col.startswith("s1dq"):  return "Aspiracion (sect1d)"
    if col in ["s1q2","s1q6","s1q8","s1q25","s1q28","zone"]: return "Demografia (sect1)"
    if col.startswith("s1eq"): return "Remesas (sect1e)"
    if col.startswith("s4aq"): return "Laboral (sect4a)"
    if col.startswith("s9q"):  return "Seg. alim. (sect9)"
    if col.startswith("wt"):   return "Pesos (secta)"
    return "Otro"

miss_df = (base.isnull().mean() * 100).reset_index()
miss_df.columns = ["variable", "pct_missing"]
miss_df["grupo"] = miss_df["variable"].apply(tag)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Seccion 4a - Valores faltantes en el dataset fusionado", fontweight="bold")

ax = axes[0]
miss_mat   = base.isnull().astype(int)
sample_idx = np.random.choice(miss_mat.index, size=min(1000, len(miss_mat)), replace=False)
sns.heatmap(miss_mat.loc[sample_idx].T,
            cmap=["#27AE60", "#E74C3C"], cbar=False,
            ax=ax, yticklabels=True, xticklabels=False, linewidths=0)
ax.set_title("Patron de missings (muestra 1,000 obs.)\n"
             "Verde = presente  |  Rojo = faltante", fontweight="bold")
ax.set_xlabel("Observaciones")
ax.tick_params(axis="y", labelsize=7)

ax2 = axes[1]
miss_with = miss_df[miss_df["pct_missing"] > 0].sort_values("pct_missing")
colors_miss = miss_with["pct_missing"].apply(
    lambda x: "#27AE60" if x < 10 else ("#F39C12" if x < 40 else "#E74C3C"))
ax2.barh(miss_with["variable"], miss_with["pct_missing"],
         color=colors_miss, alpha=0.85, height=0.65)
ax2.axvline(10, ls="--", color="#F39C12", linewidth=0.9, label="10%")
ax2.axvline(40, ls="--", color="#E74C3C", linewidth=0.9, label="40%")
ax2.set_xlabel("% valores faltantes")
ax2.set_title("% missing por variable\n"
              "Verde <10%  |  Naranja 10-40%  |  Rojo >40%", fontweight="bold")
ax2.legend(fontsize=9, frameon=False)
ax2.tick_params(axis="y", labelsize=8)
ax2.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax2.grid(axis="y", visible=False)
ax2.spines["left"].set_visible(False)
ax2.tick_params(left=False)

plt.tight_layout()
save(fig, "04a_missing_values")

[4] Dataset fusionado y missings...
  Dataset fusionado: 16,590 filas x 35 columnas
  v 04a_missing_values.png


In [9]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Seccion 4b - % missing promedio por grupo de variables", fontweight="bold")
group_miss = miss_df.groupby("grupo")["pct_missing"].mean().sort_values()
colors_g   = group_miss.apply(
    lambda x: "#27AE60" if x < 10 else ("#F39C12" if x < 40 else "#E74C3C"))
bars = ax.barh(group_miss.index, group_miss.values,
               color=colors_g, alpha=0.85, height=0.55)
ax.axvline(10, ls="--", color="#F39C12", linewidth=0.9)
ax.axvline(40, ls="--", color="#E74C3C", linewidth=0.9)
ax.set_xlabel("% missing promedio")
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f}%", va="center", fontsize=10)
plt.tight_layout()
save(fig, "04b_missing_por_grupo")

  v 04b_missing_por_grupo.png


In [10]:
print("[4c] Correlaciones...")

rename_map = {
    "target"   : "TARGET",        "s1dq3a": "dest_interno",
    "s1dq3b"   : "dest_externo",  "s1dq7"  : "intento_migrar",
    "s1eq0a"   : "remesas_ext",   "s1eq3"  : "monto_rem_ext",
    "s1eq6"    : "monto_rem_int", "s1q6"   : "edad",
    "s1q8"     : "casado",        "s1q25"  : "en_escuela",
    "s4aq1"    : "empleado_7d",   "s4aq4"  : "busca_empleo",
    "s4aq46"   : "horas_trabajo", "s4aq47" : "ingresos",
    "s9q1a"    : "HFIAS_1",       "s9q1b"  : "HFIAS_2",
    "s9q1c"    : "HFIAS_3",       "s9q1d"  : "HFIAS_4",
    "s9q1e"    : "HFIAS_5",       "s9q1f"  : "HFIAS_6",
    "s9q1g"    : "HFIAS_7",       "s9q1h"  : "HFIAS_8",
    "s9q1i"    : "HFIAS_9",       "s9q1j"  : "HFIAS_10",
    "s9q2a"    : "HFIAS_score",   "s9q4"   : "sin_comer",
}

corr_df = base.copy()
yn_cols = ["s1dq3a","s1dq3b","s1dq7","s1eq0a","s4aq1","s4aq4",
           "s9q1a","s9q1b","s9q1c","s9q1d","s9q1e",
           "s9q1f","s9q1g","s9q1h","s9q1i","s9q1j","s9q4"]
for c in yn_cols:
    if c in corr_df.columns:
        corr_df[c] = yn_to_bin(corr_df[c])

num_cols = corr_df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["hhid","indiv","wt_cross_wave5"]]
corr_mat = (corr_df[num_cols].corr()
            .rename(index=rename_map, columns=rename_map)
            .dropna(axis=0, how="all").dropna(axis=1, how="all"))

fig, ax = plt.subplots(figsize=(16, 13))
fig.suptitle("Seccion 4c - Matriz de correlacion (Pearson)", fontweight="bold")
mask      = np.triu(np.ones_like(corr_mat, dtype=bool))
# Schwabish: anotar solo |r| >= 0.10 para reducir clutter
annot_arr = corr_mat.applymap(lambda v: f"{v:.2f}" if abs(v) >= 0.10 else "")
sns.heatmap(corr_mat, mask=mask, annot=annot_arr, fmt="",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.2, annot_kws={"size": 8},
            cbar_kws={"shrink": 0.7, "label": "Correlacion de Pearson"})
ax.set_title("Triangulo inferior  |  se muestran |r| >= 0.10",
             fontsize=10, pad=10)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
save(fig, "04c_correlacion")

[4c] Correlaciones...
  v 04c_correlacion.png


In [11]:
target_corr = corr_mat["TARGET"].drop("TARGET").sort_values().dropna()

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle("Seccion 4d - Correlacion de cada variable con la aspiracion migratoria",
             fontweight="bold")
colors_bar = [ACCENT if v < 0 else SECONDARY for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors_bar, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlacion de Pearson con TARGET")
ax.set_title("Azul = correlacion positiva  |  Rojo = negativa", fontsize=9)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for v, y in zip(target_corr.values, range(len(target_corr))):
    ha  = "left"  if v >= 0 else "right"
    pad = 0.003   if v >= 0 else -0.003
    ax.text(v + pad, y, f"{v:.2f}", va="center", ha=ha, fontsize=8)
plt.tight_layout()
save(fig, "04d_corr_con_target")

  v 04d_corr_con_target.png


### SECCION 5 - Distribuciones de features clave vs. target

In [12]:
print("[5] Distribuciones por target...")

plot_df = corr_df.copy()
plot_df["target_label"] = plot_df["target"].map({1: "Desea migrar", 0: "No desea"})
COLORS_T = {"Desea migrar": SECONDARY, "No desea": ACCENT}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Seccion 5 - Distribuciones de features clave por aspiracion migratoria",
    fontweight="bold")
axes = axes.flatten()

# 5.1 Edad
ax = axes[0]
for lbl, grp in plot_df.groupby("target_label"):
    ages = grp["s1q6"].dropna()
    ages = ages[(ages >= 15) & (ages <= 80)]
    ax.hist(ages, bins=20, alpha=0.55, label=lbl, density=True,
            color=COLORS_T[lbl], edgecolor="white", linewidth=0.4)
g1 = plot_df[plot_df["target"] == 1]["s1q6"].dropna()
g0 = plot_df[plot_df["target"] == 0]["s1q6"].dropna()
_, p = stats.ttest_ind(g1, g0)
pval = "<0.001" if p < 0.001 else f"{p:.3f}"
ax.set_title(f"Distribucion de edad  (t-test p={pval})", fontweight="bold")
ax.set_xlabel("Anos")
ax.set_ylabel("Densidad")
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)

# 5.2 Score HFIAS
ax = axes[1]
hfias_rate = plot_df.groupby("s9q2a")["target"].mean() * 100
ax.bar(hfias_rate.index, hfias_rate.values, color=PALETTE, alpha=0.82, width=0.8)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(["Seguro", "Leve", "Moderado", "Severo"], fontsize=9)
ax.set_ylabel("% desea migrar")
ax.set_xlabel("Nivel de inseguridad alimentaria (HFIAS)")
ax.set_title("Aspiracion por inseguridad alimentaria", fontweight="bold")
ax.set_ylim(0, 100)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)

# 5.3 Ingresos laborales (log)
ax = axes[2]
for lbl, grp in plot_df.groupby("target_label"):
    inc = grp["s4aq47"].dropna()
    inc = inc[(inc > 0) & (inc < inc.quantile(0.99))]
    if len(inc) > 10:
        ax.hist(np.log1p(inc), bins=25, alpha=0.55, label=lbl,
                density=True, color=COLORS_T[lbl], edgecolor="white", linewidth=0.4)
ax.set_title("Log(Ingresos laborales)\n[obs. >0, sin outliers extremos]",
             fontweight="bold")
ax.set_xlabel("Log(1 + ingresos, NGN)")
ax.set_ylabel("Densidad")
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)

# 5.4 Nivel educativo
ax = axes[3]
edu_rate = (plot_df.groupby("s1q28")["target"]
            .mean().reset_index()
            .rename(columns={"target": "rate", "s1q28": "edu"}))
edu_rate["edu_short"] = edu_rate["edu"].astype(str).str.slice(0, 22)
edu_rate = edu_rate.sort_values("rate")
ax.barh(edu_rate["edu_short"], edu_rate["rate"] * 100,
        color=PALETTE, alpha=0.82, height=0.65)
ax.axvline(global_rate, ls="--", color=ACCENT, linewidth=1, alpha=0.7)
ax.set_xlabel("% desea migrar")
ax.set_title("Tasa de aspiracion por\nnivel educativo", fontweight="bold")
ax.tick_params(axis="y", labelsize=7.5)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)

# 5.5 Remesas del exterior
ax = axes[4]
remit_rate = plot_df.groupby("s1eq0a")["target"].mean()
labels_r   = ["No recibe\nremesas", "Recibe\nremesas ext."]
vals_r     = [remit_rate.get(0, np.nan), remit_rate.get(1, np.nan)]
valid      = [(l, v) for l, v in zip(labels_r, vals_r) if not np.isnan(v)]
bars       = ax.bar([x[0] for x in valid], [x[1]*100 for x in valid],
                    color=[ACCENT, SECONDARY], alpha=0.88, width=0.4)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1)
ax.set_ylim(0, 100)
ax.set_ylabel("% desea migrar")
ax.set_title("Tasa por recepcion de remesas externas", fontweight="bold")
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
label_bars_v(ax)

# 5.6 Situacion laboral (ultimos 7 dias)
ax = axes[5]
emp_rate  = plot_df.groupby("s4aq1")["target"].mean()
labels_e  = ["No trabajo", "Trabajo"]
vals_e    = [emp_rate.get(0, np.nan), emp_rate.get(1, np.nan)]
valid_e   = [(l, v) for l, v in zip(labels_e, vals_e) if not np.isnan(v)]
bars      = ax.bar([x[0] for x in valid_e], [x[1]*100 for x in valid_e],
                   color=[ACCENT, SECONDARY], alpha=0.88, width=0.4)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1)
ax.set_ylim(0, 100)
ax.set_ylabel("% desea migrar")
ax.set_title("Tasa por situacion laboral (ultimos 7 dias)", fontweight="bold")
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
label_bars_v(ax)

plt.tight_layout()
save(fig, "05_distribuciones_features")

[5] Distribuciones por target...
  v 05_distribuciones_features.png


### SECCION 6 - Shocks economicos (sect12)

In [13]:
print("[6] Shocks economicos...")

shocks_raw["affected"] = (shocks_raw["s12q1"] == "1. YES").astype(int)
shock_wide = (
    shocks_raw.pivot_table(
        index="hhid", columns="shock_cd",
        values="affected", aggfunc="max", fill_value=0
    ).reset_index()
)
shock_wide.columns.name = None
shock_wide.columns = (
    ["hhid"] + [
        c.split(". ", 1)[-1][:30] if isinstance(c, str) and ". " in c else str(c)
        for c in shock_wide.columns[1:]
    ]
)

shock_prev   = shock_wide.drop(columns="hhid").mean().sort_values(ascending=True)
shock_target = base[["hhid", "target"]].merge(shock_wide, on="hhid", how="left")
shock_target["target"] = pd.to_numeric(shock_target["target"], errors="coerce")
shock_corr = (
    shock_target.drop(columns=["hhid"]).corr()["target"].drop("target").sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle(
    "Seccion 6a - Shocks economicos: prevalencia y correlacion con aspiraciones",
    fontweight="bold")

ax = axes[0]
colors_shock = [ACCENT if v > 0.15 else SECONDARY for v in shock_prev.values]
bars = ax.barh(shock_prev.index, shock_prev.values * 100,
               color=colors_shock, alpha=0.85, height=0.65)
ax.set_xlabel("% hogares afectados")
ax.set_title("Prevalencia de shocks\n(rojo = afecta a >15% de hogares)",
             fontweight="bold")
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
for bar in bars:
    if bar.get_width() > 1:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f"{bar.get_width():.1f}%", va="center", fontsize=8)

ax = axes[1]
colors_c = [SECONDARY if v >= 0 else ACCENT for v in shock_corr.values]
ax.barh(shock_corr.index, shock_corr.values, color=colors_c, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlacion con aspiracion migratoria")
ax.set_title("Correlacion de cada shock con el TARGET\n"
             "Azul = positiva  |  Rojo = negativa", fontweight="bold")
ax.tick_params(axis="y", labelsize=8)
ax.grid(axis="x", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)

plt.tight_layout()
save(fig, "06a_shocks_economicos")

[6] Shocks economicos...
  v 06a_shocks_economicos.png


In [14]:
shock_wide_sub             = shock_wide.copy()
shock_wide_sub["n_shocks"] = shock_wide_sub.drop(columns="hhid").sum(axis=1)
shock_index = (
    base[["hhid", "target"]]
    .merge(shock_wide_sub[["hhid", "n_shocks"]], on="hhid", how="left")
)
shock_agg = (shock_index.groupby("n_shocks")["target"]
             .agg(["mean", "count"]).reset_index())

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Seccion 6b - A mas shocks acumulados, mayor aspiracion migratoria?",
             fontweight="bold")
bars = ax.bar(shock_agg["n_shocks"], shock_agg["mean"] * 100,
              color=PALETTE, alpha=0.82, width=0.7)
ax.axhline(global_rate, ls="--", color=GRAY, linewidth=1,
           label=f"Media global ({global_rate:.1f}%)")
ax.set_xlabel("Numero de shocks distintos que afectaron al hogar")
ax.set_ylabel("% desea migrar")
ax.set_ylim(0, 100)
ax.legend(fontsize=9, frameon=False)
ax.grid(axis="y", color="#E8E8E8", linewidth=0.6)
ax.grid(axis="x", visible=False)
for bar, (_, row) in zip(bars, shock_agg.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{row['mean']*100:.1f}%", ha="center", va="bottom",
            fontsize=8.5, fontweight="bold")
    if row["count"] > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, 2,
                f"n={int(row['count'])}", ha="center", va="bottom",
                fontsize=7, color="white", fontweight="bold")
plt.tight_layout()
save(fig, "06b_shock_index")

  v 06b_shock_index.png


### SECCION 7 - Tabla resumen para decisiones de limpieza

In [15]:
print("[7] Tabla resumen de decisiones...")

summary_rows = []
for col in base.columns:
    if col in ["hhid", "indiv"]:
        continue
    miss  = base[col].isnull().mean() * 100
    dtype = str(base[col].dtype)
    nuniq = base[col].nunique(dropna=True)
    if miss > 40:
        rec = "REVISAR - >40% missing"
    elif miss > 10:
        rec = "Imputar (mediana/moda)"
    elif nuniq <= 1:
        rec = "DROP - varianza cero"
    elif col.endswith("_os"):
        rec = "DROP - campo open-ended"
    else:
        rec = "Usar"
    summary_rows.append({
        "Variable"    : col,
        "Interpretado": rename_map.get(col, col),
        "Grupo"       : tag(col),
        "% Missing"   : round(miss, 1),
        "Dtype"       : dtype,
        "N unicos"    : nuniq,
        "Rec."        : rec,
    })

summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(f"{OUT}/07_summary_tabla.csv", index=False)
print("  OK 07_summary_tabla.csv")

n_per_page = 30
pages = [summary_table.iloc[i:i + n_per_page]
         for i in range(0, len(summary_table), n_per_page)]

REC_COLORS = {
    "Usar"                  : "#D5F5E3",
    "DROP - varianza cero"  : "#FADBD8",
    "DROP - campo open-ended": "#FADBD8",
    "REVISAR - >40% missing": "#FDEBD0",
    "Imputar (mediana/moda)": "#D6EAF8",
}

for pg_idx, page in enumerate(pages):
    fig, ax = plt.subplots(figsize=(20, len(page) * 0.45 + 2))
    fig.suptitle(
        f"Seccion 7 - Resumen de variables para data cleaning "
        f"(pag. {pg_idx+1}/{len(pages)})",
        fontsize=13, fontweight="bold")
    ax.axis("off")
    tbl = ax.table(cellText=page.values, colLabels=page.columns,
                   loc="center", cellLoc="left")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1, 1.5)
    for j in range(len(page.columns)):
        tbl[0, j].set_facecolor(PALETTE)
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    rec_idx = list(page.columns).index("Rec.")
    for i in range(1, len(page) + 1):
        val   = page["Rec."].iloc[i - 1]
        color = REC_COLORS.get(val, "white")
        tbl[i, rec_idx].set_facecolor(color)
    plt.tight_layout()
    save(fig, f"07_summary_tabla_p{pg_idx+1}")

[7] Tabla resumen de decisiones...
  OK 07_summary_tabla.csv
  v 07_summary_tabla_p1.png
  v 07_summary_tabla_p2.png


### Resumen final

In [16]:
print("\n" + "="*60)
print("RESUMEN DEL DIAGNOSTICO")
print("="*60)
n_total = base.shape[0]
n_complete = base.dropna().shape[0]
print(f"  Dataset fusionado : {n_total:,} filas x {base.shape[1]} columnas")
print(f"  Target YES        : {base['target'].mean()*100:.1f}%")
print(f"  Sin ningun missing: {n_complete:,} obs ({n_complete/n_total*100:.1f}%)")
print(f"  Vars sin missings : {(base.isnull().sum()==0).sum()}")
print(f"  Vars >40% missing : {(base.isnull().mean()>0.4).sum()}")
miss_mid = ((base.isnull().mean()>0.1)&(base.isnull().mean()<=0.4)).sum()
print(f"  Vars 10-40% miss. : {miss_mid}")
print(f"  Vars <10% miss.   : {(base.isnull().mean()<0.1).sum()}")
print(f"\nArchivos generados en: {OUT}/")
print("="*60)


RESUMEN DEL DIAGNOSTICO
  Dataset fusionado : 16,590 filas x 35 columnas
  Target YES        : 71.0%
  Sin ningun missing: 0 obs (0.0%)
  Vars sin missings : 21
  Vars >40% missing : 8
  Vars 10-40% miss. : 3
  Vars <10% miss.   : 24

Archivos generados en: /Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/Proyecto_poster/diagnostico/
